In [0]:
-- ─── dim_client ────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE hive_metastore.gold.dim_client
USING DELTA
LOCATION 'abfss://gold@adlscommercialprod.dfs.core.windows.net/dim_client'
AS
SELECT
    DENSE_RANK() OVER (ORDER BY client_id) AS sk_client,
    client_id,
    MAX(nom_client)  AS nom_client,
    MAX(region)      AS region
FROM hive_metastore.silver.ventes_clean
GROUP BY client_id;

-- ─── dim_produit ────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE hive_metastore.gold.dim_produit
USING DELTA
LOCATION 'abfss://gold@adlscommercialprod.dfs.core.windows.net/dim_produit'
AS
SELECT
    DENSE_RANK() OVER (ORDER BY produit_id) AS sk_produit,
    produit_id,
    MAX(produit)      AS nom_produit,
    MAX(categorie)    AS categorie,
    MAX(segment_prix) AS segment_prix
FROM hive_metastore.silver.ventes_clean
GROUP BY produit_id;

-- ─── dim_date ────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE hive_metastore.gold.dim_date
USING DELTA
LOCATION 'abfss://gold@adlscommercialprod.dfs.core.windows.net/dim_date'
AS
SELECT
    DATE_FORMAT(date_commande, 'yyyyMMdd')    AS sk_date,
    date_commande,
    YEAR(date_commande)                        AS annee,
    QUARTER(date_commande)                     AS trimestre,
    MONTH(date_commande)                       AS mois,
    DATE_FORMAT(date_commande, 'EEEE')        AS nom_jour,
    WEEKOFYEAR(date_commande)                  AS semaine
FROM hive_metastore.silver.ventes_clean
GROUP BY date_commande;

num_affected_rows,num_inserted_rows


In [0]:
%python
from pyspark.sql.functions import monotonically_increasing_id, date_format

# Unity Catalog : notation 3 niveaux
df_silver   = spark.table("hive_metastore.silver.ventes_clean")
dim_client  = spark.table("hive_metastore.gold.dim_client")
dim_produit = spark.table("hive_metastore.gold.dim_produit")

df_fact = (
    df_silver
    .join(dim_client.select("client_id", "sk_client"),
          on="client_id", how="left")
    .join(dim_produit.select("produit_id", "sk_produit"),
          on="produit_id", how="left")
    .withColumn("sk_date",  date_format("date_commande", "yyyyMMdd"))
    .withColumn("sk_vente", monotonically_increasing_id())
    .select("sk_vente", "order_id", "sk_client",
            "sk_produit", "sk_date", "qte",
            "prix_unitaire", "remise", "montant_brut", "montant_net")
)

(
    df_fact.write.format("delta")
    .mode("overwrite")
    .partitionBy("sk_date")
    .option("path", "abfss://gold@adlscommercialprod.dfs.core.windows.net/fact_ventes")
    .saveAsTable("hive_metastore.gold.fact_ventes")
)

print(f"✅ fact_ventes : {df_fact.count()} lignes")
df_fact.show()

✅ fact_ventes : 8 lignes
+--------+--------+---------+----------+--------+---+-------------+------+------------+-----------+
|sk_vente|order_id|sk_client|sk_produit| sk_date|qte|prix_unitaire|remise|montant_brut|montant_net|
+--------+--------+---------+----------+--------+---+-------------+------+------------+-----------+
|       0|       1|        1|         1|20240115|  2|       1200.0|   0.1|      2400.0|     2160.0|
|       1|       6|        4|         2|20240314| 10|        89.99|   0.2|       899.9|     719.92|
|       2|       3|        1|         3|20240203|  1|        699.0|  0.05|       699.0|     664.05|
|       3|       5|        2|         4|20240301|  2|       149.99|   0.0|      299.98|     299.98|
|       4|       4|        3|         1|20240217|  3|       1200.0|  0.15|      3600.0|     3060.0|
|       5|       8|        3|         3|20240405|  2|        699.0|   0.1|      1398.0|     1258.2|
|       6|       7|        1|         5|20240328|  4|        199.0|  0.05| 